In [9]:
import torch
import torch.nn as nn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ──────────────────────────────────────────────
# 1. Perceptron from scratch
# ──────────────────────────────────────────────
class Perceptron(nn.Module):
    """Single-layer perceptron for binary classification."""

    def __init__(self, n_features: int):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)  # one output neuron

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(self.linear(x))     # squash to [0, 1]

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            return (self.forward(x) >= 0.5).float()


# ──────────────────────────────────────────────
# 2. Example usage
# ──────────────────────────────────────────────
if __name__ == "__main__":

    # --- Generate a toy dataset ---
    X, y = make_classification(
        n_samples=500, n_features=4, n_informative=3,
        n_redundant=0, random_state=42,
    )

    # Scale & split
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42,
    )

    # NumPy → Tensors
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_test  = torch.tensor(X_test,  dtype=torch.float32)
    y_test  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

    # --- Create model, loss, optimizer ---
    model     = Perceptron(n_features=4)
    criterion = nn.BCELoss()            # binary cross-entropy
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

    # --- Train ---
    EPOCHS = 50000
    for epoch in range(1, EPOCHS + 1):
        # forward
        y_pred = model(X_train)
        loss   = criterion(y_pred, y_train)

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            acc = (model.predict(X_train) == y_train).float().mean()
            print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Train Acc: {acc:.2%}")

    # --- Evaluate ---
    test_acc = (model.predict(X_test) == y_test).float().mean()
    print(f"\nTest Accuracy: {test_acc:.2%}")

    # --- Inspect learned weights ---
    w = model.linear.weight.data.squeeze()
    b = model.linear.bias.data.item()
    print(f"Weights: {w.tolist()}")
    print(f"Bias:    {b:.4f}")

Epoch  10 | Loss: 0.6275 | Train Acc: 78.00%
Epoch  20 | Loss: 0.5135 | Train Acc: 84.00%
Epoch  30 | Loss: 0.4551 | Train Acc: 86.25%
Epoch  40 | Loss: 0.4210 | Train Acc: 87.00%
Epoch  50 | Loss: 0.3990 | Train Acc: 86.50%
Epoch  60 | Loss: 0.3837 | Train Acc: 87.00%
Epoch  70 | Loss: 0.3725 | Train Acc: 87.75%
Epoch  80 | Loss: 0.3640 | Train Acc: 88.00%
Epoch  90 | Loss: 0.3572 | Train Acc: 88.25%
Epoch 100 | Loss: 0.3518 | Train Acc: 88.25%
Epoch 110 | Loss: 0.3473 | Train Acc: 87.75%
Epoch 120 | Loss: 0.3436 | Train Acc: 87.75%
Epoch 130 | Loss: 0.3404 | Train Acc: 87.75%
Epoch 140 | Loss: 0.3377 | Train Acc: 87.75%
Epoch 150 | Loss: 0.3354 | Train Acc: 87.75%
Epoch 160 | Loss: 0.3333 | Train Acc: 87.75%
Epoch 170 | Loss: 0.3315 | Train Acc: 87.75%
Epoch 180 | Loss: 0.3300 | Train Acc: 87.75%
Epoch 190 | Loss: 0.3286 | Train Acc: 87.75%
Epoch 200 | Loss: 0.3273 | Train Acc: 87.75%
Epoch 210 | Loss: 0.3262 | Train Acc: 87.75%
Epoch 220 | Loss: 0.3252 | Train Acc: 87.75%
Epoch 230 